# 10年定着予測 - モデル多様性の等重み平均（43_）

**背景**: モデリング側の手はほぼ尽きた（`submit_result_report.md` 第65節）。
`14_` 以降ずっと **CatBoost単体 + シード平均** で来ており、
**モデルファミリの多様性だけが未検証の手**として残っている。

| 手 | 結果 |
|---|---|
| Train全件で再学習 | ✅ −0.00646 |
| 冗長列の削減（441→113列） | ✅ −0.00093 |
| シード平均 | ✅ −0.00034 |
| 反復数・ハイパーパラメータ再探索 | ❌ 2度不発 |
| さらなる減量（113→49→33列） | ❌ +0.0042 / +0.0098 |
| 学習母集団の是正 | ❌ +0.00176 |
| 新規特徴量（EDA v1〜v6・文献9件） | ❌ 全滅 |
| **モデル多様性の等重み平均** | **⬜ 本ノートブック** |

## 重みは学習しない

`10_`〜`13_` でアンサンブルを試したときの教訓（[ensemble-oof-overfitting]）:

- `12_` の Stacking / Optimized Weighted Average は OOF を上回ったが **Public では負けた**
- `13_` の AutoGluon Weighted Ensemble（貪欲法で重みを学習）は Public **0.575345**、
  同じ実行の **CatBoost単体が 0.574165 で勝っている**

**OOFから重みを学習すると過学習する**という結論は変わっていない。
本ノートブックは**等重み固定**にする。これはシード平均と同じ理屈であり、
シード平均は Public で無害と確認済み（−0.00034）。

## 期待値について（正直に）

過去の実績は芳しくない（13_でアンサンブルが単体に0.0012負け）。
ただしあれは**重み学習あり・441列以前の特徴量・11モデル**という別条件だった。
期待値は **−0.000〜0.004** 程度で、1位との差0.0209は埋まらない。

## 事前チェック: 多様性が無ければ平均する意味も無い

第C節で **CatBoost と LightGBM / XGBoost の予測相関**を測る。
相関が 0.99 を超えるなら実質同じモデルであり、平均しても何も起きない。
**この相関は学習直後に分かるので、提出前に見込みが立つ。**

## 実行構成

ベースラインは `40_` R6_lean と同一の113列。全モデルで同じ特徴量・同じ5シード。

| config | 内容 |
|---|---|
| `E0_catboost` | CatBoost単体（`A_PARAMS`, 560反復）＝ **R6_lean の再現**。参照用 |
| `E1_lgbm` | LightGBM単体（マルチシードOptunaで探索） |
| `E2_xgb` | XGBoost単体（同上） |
| `E3_cat_lgbm` | **CatBoost + LightGBM の等重み平均** |
| `E4_all3` | ゲートを通ったモデル全部の等重み平均 |

## ゲート（事前登録）

弱いモデルを混ぜると平均が引きずられるので、**単体検証がCatBoostから0.01以内**の
モデルだけをアンサンブルに入れる。0.01 は `39_` で校正した検証の分解能（CI半幅±0.011）に由来する。

`E1`・`E2`（単体）は**提出しない**。CatBoost単体に勝つことは期待しておらず、
アンサンブルの材料としての質を測るためだけに学習する。

## 判定（事前登録）

- 採否は **Public のみ**で決める
- `E3`・`E4` のうち、`E0` との予測平均絶対差が **ノイズ床（`41_` 実測: 平均0.01246・95%上限0.02122）**
  を超えたものだけ提出する。超えなければ提出しても情報が得られない（`38_` H1・`41_` X1 と同じ判断）

## 実行環境

Google Colab Pro の **CPUハイメモリ**。113列なので軽い。
Optuna 2スタディ（各25試行×3シード）＋最終学習30回で想定 **1〜1.5時間**。

> ⚠️ **ローカルMacで先行実行しないこと。** `27_` のチェックポイント同期事故を避ける。


In [1]:
!pip install -q catboost optuna lightgbm xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 29.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.1 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "43_model_diversity_equal_weight"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-12 11:28:35] [INFO] === [43_model_diversity_equal_weight] 実験開始 ===


INFO:43_model_diversity_equal_weight:=== [43_model_diversity_equal_weight] 実験開始 ===


[2026-08-12 11:28:36] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


INFO:43_model_diversity_equal_weight:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


[2026-08-12 11:28:36] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/43_model_diversity_equal_weight_checkpoint.csv


INFO:43_model_diversity_equal_weight:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/43_model_diversity_equal_weight_checkpoint.csv


[2026-08-12 11:28:36] [INFO] チェックポイントは未作成（新規実行）


INFO:43_model_diversity_equal_weight:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-12 11:28:41] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:43_model_diversity_equal_weight:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-12 11:28:41] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:43_model_diversity_equal_weight:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-12 11:28:41] [INFO] 定着率: 0.5647


INFO:43_model_diversity_equal_weight:定着率: 0.5647


[2026-08-12 11:28:41] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:43_model_diversity_equal_weight:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-12 11:28:41] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:43_model_diversity_equal_weight:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-12 11:28:41] [INFO] Test  早期退職者: 0名 / 2502名


INFO:43_model_diversity_equal_weight:Test  早期退職者: 0名 / 2502名


[2026-08-12 11:28:41] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:43_model_diversity_equal_weight:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-12 11:28:41] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:43_model_diversity_equal_weight:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-12 11:28:42] [INFO] ------------------------------------------------------------


INFO:43_model_diversity_equal_weight:------------------------------------------------------------


[2026-08-12 11:28:42] [INFO] split非依存の基本特徴量を生成中...


INFO:43_model_diversity_equal_weight:split非依存の基本特徴量を生成中...


[2026-08-12 11:28:42] [INFO] ------------------------------------------------------------


INFO:43_model_diversity_equal_weight:------------------------------------------------------------


[2026-08-12 11:34:08] [INFO] split非依存の基本特徴量生成完了


INFO:43_model_diversity_equal_weight:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-12 11:34:08] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:43_model_diversity_equal_weight:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-12 11:34:10] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:43_model_diversity_equal_weight:入社時メモ: SVD累積寄与率=0.760


[2026-08-12 11:34:13] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:43_model_diversity_equal_weight:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-12 11:34:15] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:43_model_diversity_equal_weight:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-12 11:34:15] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:43_model_diversity_equal_weight:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-12 11:34:15] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:43_model_diversity_equal_weight:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-12 11:36:11] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:43_model_diversity_equal_weight:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-12 11:36:12] [INFO] Persona単位の基本特徴量を生成中...


INFO:43_model_diversity_equal_weight:Persona単位の基本特徴量を生成中...


[2026-08-12 11:36:12] [INFO] Persona単位の基本特徴量処理完了


INFO:43_model_diversity_equal_weight:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-12 11:36:12] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:43_model_diversity_equal_weight:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-12 11:36:12] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:43_model_diversity_equal_weight:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-12 11:36:12] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:43_model_diversity_equal_weight:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [15]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [16]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [17]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-12 11:36:13] [INFO] ============================================================


INFO:43_model_diversity_equal_weight:============================================================


[2026-08-12 11:36:13] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:43_model_diversity_equal_weight:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-12 11:36:13] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:43_model_diversity_equal_weight:B/C: train=2208, val=535（生存者のみ）


[2026-08-12 11:36:13] [INFO] D: train=2761（全件）, val=0（空）


INFO:43_model_diversity_equal_weight:D: train=2761（全件）, val=0（空）


[2026-08-12 11:36:13] [INFO] 特徴量数: 441


INFO:43_model_diversity_equal_weight:特徴量数: 441


## 10. 特徴量グループの棚卸し（`40_` から移植）

In [18]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


In [19]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## 11. ベースライン（113列）

In [20]:
# ============================================================
# ベースライン: 40_ R6_lean と同一の113列
# ============================================================

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}
BASE_SPEC = {"groups": CORE_GROUPS, "agg_stats": AGG_KEEP_STATS}


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


FEATS = cols_for(BASE_SPEC, ag_train_80b)
assert FEATS == cols_for(BASE_SPEC, ag_full), "80%学習と全件学習で列が食い違っている"
assert len(FEATS) == 113, f"{len(FEATS)}列（40_ R6_lean と同じ113列のはず）"

CAT_COLS = [c for c in FEATS if ag_full[c].dtype == "object"]
NUM_COLS = [c for c in FEATS if c not in CAT_COLS]
print(f"特徴量 {len(FEATS)} 列（カテゴリ {len(CAT_COLS)} / 数値 {len(NUM_COLS)}）")
print(f"  カテゴリ列: {CAT_COLS}")
print("✅ 40_ R6_lean と同一の113列")


特徴量 113 列（カテゴリ 8 / 数値 105）
  カテゴリ列: ['入社区分', '専攻分野', '採用経路', '性別', '初期職種', '初期勤務地', '初期役割', '転居x勤務地_状態_v2']
✅ 40_ R6_lean と同一の113列


## 12. LightGBM / XGBoost 用のデータ準備

In [21]:
# ============================================================
# LightGBM / XGBoost 用のデータ準備
#   CatBoost はカテゴリを文字列のまま扱えるが、LightGBM/XGBoost は
#   pandas の category dtype が必要。学習側とTest側でカテゴリ集合を揃える。
# ============================================================

def _to_frame(df):
    return df[FEATS].copy()


def make_cat_frames(*frames):
    """複数フレームのカテゴリ列を、共通のカテゴリ集合を持つ category dtype に変換する。

    学習側に無い値がTestに出た場合、揃えないとエンコードがずれる。
    数値列の欠損は CatBoost と同じ -999 で埋める（3モデルで前処理を揃えるため）。
    """
    outs = [_to_frame(f) for f in frames]
    for c in CAT_COLS:
        cats = sorted(set().union(*[set(o[c].dropna().unique()) for o in outs]))
        for o in outs:
            o[c] = pd.Categorical(o[c], categories=cats)
    for o in outs:
        o[NUM_COLS] = o[NUM_COLS].fillna(-999)
    return outs


Xtr_c, Xva_c, Xfull_c, Xte_c = make_cat_frames(ag_train_80b, ag_val_surv, ag_full, test_features_full)
ytr = ag_train_80b[TARGET_COL].values
yva = ag_val_surv[TARGET_COL].values
yfull = ag_full[TARGET_COL].values

# CatBoost は従来どおり（文字列のまま・-999埋め）
Xtr_cb = ag_train_80b[FEATS].fillna(-999)
Xva_cb = ag_val_surv[FEATS].fillna(-999)
Xfull_cb = ag_full[FEATS].fillna(-999)
Xte_cb = test_features_full[FEATS].fillna(-999)

for nm, f in [("学習80%", Xtr_c), ("検証", Xva_c), ("全件", Xfull_c), ("Test", Xte_c)]:
    print(f"  {nm:<8s}: {f.shape}  category列 {sum(str(f[c].dtype)=='category' for c in f.columns)}")

# カテゴリ集合が全フレームで一致していること
for c in CAT_COLS:
    cs = [tuple(f[c].cat.categories) for f in (Xtr_c, Xva_c, Xfull_c, Xte_c)]
    assert len(set(cs)) == 1, f"{c}: フレーム間でカテゴリ集合が不一致"
print("✅ カテゴリ集合を全フレームで統一")


  学習80%   : (2208, 113)  category列 8
  検証      : (535, 113)  category列 8
  全件      : (2761, 113)  category列 8
  Test    : (2502, 113)  category列 8
✅ カテゴリ集合を全フレームで統一


## 13. 設定の事前登録

In [22]:
# ============================================================
# 設定の事前登録
# ============================================================

A_PARAMS = {
    "depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
    "border_count": 218, "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
ITER_CB = 560          # 40_ R6_lean と同一（42_で113列でも最適と確認済み）

SEEDS_SUB = [42, 2024, 7, 1234, 99]                      # D3・40_・42_ と同一
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]
TUNE_SEEDS = [42, 2024, 7]
N_TRIALS = 25

# ゲート: 単体検証が CatBoost から этого以内のモデルだけアンサンブルに入れる
GATE_MARGIN = 0.01     # 39_で校正した検証の分解能（CI半幅±0.011）に由来

# 提出ゲート: E0 との予測平均絶対差がノイズ床を超えないものは提出しない
NOISE_FLOOR_MEAN = 0.01246   # 41_ 実測（113列・5シード平均どうし）
NOISE_FLOOR_P95  = 0.02122

print(f"ゲート（アンサンブル参加）: 単体valが CatBoost + {GATE_MARGIN} 以内")
print(f"ゲート（提出）            : E0との平均絶対差 > {NOISE_FLOOR_P95}（41_実測のノイズ床95%上限）")


ゲート（アンサンブル参加）: 単体valが CatBoost + 0.01 以内
ゲート（提出）            : E0との平均絶対差 > 0.02122（41_実測のノイズ床95%上限）


## 14. 3ファミリの学習関数

In [23]:
# ============================================================
# 3ファミリの学習関数
#   前処理・シード・特徴量はすべて共通。違うのはモデルだけ。
# ============================================================
import lightgbm as lgb
import xgboost as xgb

def fit_catboost(Xtr, y, Xpred_list, params, n_iter, seeds):
    out = [[] for _ in Xpred_list]
    for s in seeds:
        m = cb.CatBoostClassifier(**params, iterations=int(n_iter), random_seed=s,
                                  verbose=False, cat_features=CAT_COLS, task_type="CPU")
        m.fit(Xtr, y)
        for i, Xp in enumerate(Xpred_list):
            out[i].append(m.predict_proba(Xp)[:, 1])
    return [np.array(o) for o in out]


def fit_lgbm(Xtr, y, Xpred_list, params, n_iter, seeds):
    out = [[] for _ in Xpred_list]
    for s in seeds:
        m = lgb.LGBMClassifier(**params, n_estimators=int(n_iter), random_state=s,
                               verbose=-1, n_jobs=-1)
        m.fit(Xtr, y, categorical_feature=CAT_COLS)
        for i, Xp in enumerate(Xpred_list):
            out[i].append(m.predict_proba(Xp)[:, 1])
    return [np.array(o) for o in out]


def fit_xgb(Xtr, y, Xpred_list, params, n_iter, seeds):
    out = [[] for _ in Xpred_list]
    for s in seeds:
        m = xgb.XGBClassifier(**params, n_estimators=int(n_iter), random_state=s,
                              enable_categorical=True, tree_method="hist",
                              eval_metric="logloss", n_jobs=-1)
        m.fit(Xtr, y)
        for i, Xp in enumerate(Xpred_list):
            out[i].append(m.predict_proba(Xp)[:, 1])
    return [np.array(o) for o in out]


def _score(vps, y):
    singles = [log_loss(y, v) for v in vps]
    return {"val_seedavg": float(log_loss(y, vps.mean(axis=0))),
            "val_single_mean": float(np.mean(singles)),
            "val_single_sd": float(np.std(singles))}


print("✅ 3ファミリの学習関数を定義（前処理・シード・特徴量は共通）")


✅ 3ファミリの学習関数を定義（前処理・シード・特徴量は共通）


## 15. LightGBM / XGBoost の探索

In [24]:
# ============================================================
# LightGBM / XGBoost のハイパーパラメータ探索（マルチシード目的関数）
#   CatBoost の再探索は 38_・42_ で不発と確定しているが、
#   LGBM/XGB はこの特徴量セットで一度も学習していないので、
#   何らかの設定は決めなければならない。38_ と同じマルチシード方式を使う。
# ============================================================

def tune_family(kind, n_trials=N_TRIALS, tune_seeds=TUNE_SEEDS):
    def objective(trial):
        if kind == "lgbm":
            params = {
                "num_leaves": trial.suggest_int("num_leaves", 4, 64, log=True),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
                "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
                "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }
            n_it = trial.suggest_int("n_estimators", 100, 1200, log=True)
            fit = fit_lgbm
        else:
            params = {
                "max_depth": trial.suggest_int("max_depth", 2, 8),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }
            n_it = trial.suggest_int("n_estimators", 100, 1200, log=True)
            fit = fit_xgb
        (vps,) = fit(Xtr_c, ytr, [Xva_c], params, n_it, tune_seeds)
        return float(np.mean([log_loss(yva, v) for v in vps]))

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    bp = dict(study.best_params)
    n_it = bp.pop("n_estimators")
    logger.info(f"[{kind}] best_value={study.best_value:.6f} n_estimators={n_it} params={bp}")
    return bp, n_it, float(study.best_value)


TUNED_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_tuned.json"
if TUNED_PATH.exists():
    _t = json.loads(TUNED_PATH.read_text())
    logger.info("探索結果をチェックポイントから復元")
else:
    _t = {}
    for _k in ["lgbm", "xgb"]:
        logger.info(f"[Optuna] {_k}: {N_TRIALS}試行 × {len(TUNE_SEEDS)}シード")
        _p, _n, _v = tune_family(_k)
        _t[_k] = {"params": _p, "n_estimators": _n, "value": _v}
    TUNED_PATH.write_text(json.dumps(_t, ensure_ascii=False))

LGBM_PARAMS, LGBM_ITER = _t["lgbm"]["params"], _t["lgbm"]["n_estimators"]
XGB_PARAMS, XGB_ITER = _t["xgb"]["params"], _t["xgb"]["n_estimators"]
print(f"LightGBM: n_estimators={LGBM_ITER}, {LGBM_PARAMS}")
print(f"XGBoost : n_estimators={XGB_ITER}, {XGB_PARAMS}")


[2026-08-12 11:36:17] [INFO] [Optuna] lgbm: 25試行 × 3シード


INFO:43_model_diversity_equal_weight:[Optuna] lgbm: 25試行 × 3シード


[2026-08-12 11:37:12] [INFO] [lgbm] best_value=0.547979 n_estimators=512 params={'num_leaves': 45, 'learning_rate': 0.01146817859176391, 'min_child_samples': 49, 'feature_fraction': 0.40067740823864223, 'bagging_fraction': 0.758257885021379, 'bagging_freq': 4, 'reg_lambda': 1.4272808002832233}


INFO:43_model_diversity_equal_weight:[lgbm] best_value=0.547979 n_estimators=512 params={'num_leaves': 45, 'learning_rate': 0.01146817859176391, 'min_child_samples': 49, 'feature_fraction': 0.40067740823864223, 'bagging_fraction': 0.758257885021379, 'bagging_freq': 4, 'reg_lambda': 1.4272808002832233}


[2026-08-12 11:37:12] [INFO] [Optuna] xgb: 25試行 × 3シード


INFO:43_model_diversity_equal_weight:[Optuna] xgb: 25試行 × 3シード


[2026-08-12 11:39:00] [INFO] [xgb] best_value=0.541626 n_estimators=133 params={'max_depth': 5, 'learning_rate': 0.0350744711929584, 'min_child_weight': 0.052371118038571224, 'subsample': 0.6531221286499025, 'colsample_bytree': 0.836607323744935, 'reg_lambda': 0.7472072646996676}


INFO:43_model_diversity_equal_weight:[xgb] best_value=0.541626 n_estimators=133 params={'max_depth': 5, 'learning_rate': 0.0350744711929584, 'min_child_weight': 0.052371118038571224, 'subsample': 0.6531221286499025, 'colsample_bytree': 0.836607323744935, 'reg_lambda': 0.7472072646996676}


LightGBM: n_estimators=512, {'num_leaves': 45, 'learning_rate': 0.01146817859176391, 'min_child_samples': 49, 'feature_fraction': 0.40067740823864223, 'bagging_fraction': 0.758257885021379, 'bagging_freq': 4, 'reg_lambda': 1.4272808002832233}
XGBoost : n_estimators=133, {'max_depth': 5, 'learning_rate': 0.0350744711929584, 'min_child_weight': 0.052371118038571224, 'subsample': 0.6531221286499025, 'colsample_bytree': 0.836607323744935, 'reg_lambda': 0.7472072646996676}


## A. 単体学習と多様性チェック

In [25]:
# ============================================================
# 第A節: 3ファミリの単体学習と多様性チェック
# ============================================================

FAMILIES = {
    "catboost": (fit_catboost, Xtr_cb, Xva_cb, Xfull_cb, Xte_cb, A_PARAMS, ITER_CB),
    "lgbm":     (fit_lgbm,     Xtr_c,  Xva_c,  Xfull_c,  Xte_c,  LGBM_PARAMS, LGBM_ITER),
    "xgb":      (fit_xgb,      Xtr_c,  Xva_c,  Xfull_c,  Xte_c,  XGB_PARAMS,  XGB_ITER),
}

# 予測は npy に保存し、既にあれば再学習しない（1〜1.5時間かかるので中断に備える）
PRED_DIR = CHECKPOINT_DIR / SCRIPT_NAME
PRED_DIR.mkdir(parents=True, exist_ok=True)

SINGLE = {}
for name, (fit, Xtr, Xva, Xfull, Xte, params, n_it) in FAMILIES.items():
    vpath = PRED_DIR / f"{name}_valpreds.npy"
    tpath = PRED_DIR / f"{name}_testpreds.npy"
    if vpath.exists() and tpath.exists():
        vps, tps = np.load(vpath), np.load(tpath)
        assert vps.shape == (len(SEEDS_VAL), len(yva)), f"{name}: 保存済みvalpredsの形が不正 {vps.shape}"
        assert tps.shape == (len(SEEDS_SUB), len(test_features_full)), \
            f"{name}: 保存済みtestpredsの形が不正 {tps.shape}"
        logger.info(f"[{name}] 保存済み予測から復元（再学習しない）")
    else:
        logger.info("=" * 60)
        logger.info(f"[{name}] 単体学習（検証8シード + 全件5シード）")
        (vps,) = fit(Xtr, ytr, [Xva], params, n_it, SEEDS_VAL)
        (tps,) = fit(Xfull, yfull, [Xte], params, n_it, SEEDS_SUB)
        np.save(vpath, vps)
        np.save(tpath, tps)
    sc = _score(vps, yva)
    SINGLE[name] = {"val_preds": vps.mean(axis=0), "test_preds": tps.mean(axis=0),
                    "test_by_seed": tps, **sc}
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{name}_valpreds.npy", vps)
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{name}_testpreds.npy", tps)
    logger.info(f"  val シード平均 {sc['val_seedavg']:.6f} / 単一 {sc['val_single_mean']:.6f} "
                f"± {sc['val_single_sd']:.6f}")

print(f"{'model':<10s}{'val(8シード)':>14s}{'単一平均':>11s}{'単一sd':>10s}{'CatBoost差':>12s}{'ゲート':>8s}")
print("-" * 66)
cb_val = SINGLE["catboost"]["val_seedavg"]
GATE_PASS = []
for name, r in SINGLE.items():
    d = r["val_seedavg"] - cb_val
    ok = (name == "catboost") or (d <= GATE_MARGIN)
    if ok:
        GATE_PASS.append(name)
    print(f"{name:<10s}{r['val_seedavg']:>14.6f}{r['val_single_mean']:>11.6f}"
          f"{r['val_single_sd']:>10.6f}{d:>+12.6f}{'✅' if ok else '❌':>8s}")
print(f"\nゲート通過: {GATE_PASS}")

# 42_ の教訓: 事前登録した閾値が僅差で決まると、その判定に意味が無くなる。
# 境界の近くに落ちた場合は明示して、閾値以外の根拠で判断できるようにする。
for _n, _r in SINGLE.items():
    if _n == "catboost":
        continue
    _d = _r["val_seedavg"] - cb_val
    if abs(_d - GATE_MARGIN) < 0.001:
        print(f"  ⚠️ {_n}: CatBoost差 {_d:+.6f} はゲート {GATE_MARGIN} の境界から "
              f"{abs(_d - GATE_MARGIN):.6f} しか離れていない。")
        print(f"     この判定は実質コイン投げ。多様性チェック（相関）と併せて判断すること。")

print()
print("=" * 66)
print("多様性チェック: ファミリ間のTest予測相関")
print("=" * 66)
names = list(SINGLE)
corr = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        corr.loc[a, b] = np.corrcoef(SINGLE[a]["test_preds"], SINGLE[b]["test_preds"])[0, 1]
print(corr.round(5).to_string())
print()
print("参考: CatBoost内でシードを引き直したときの相関（多様性ゼロの基準）")
_cb = SINGLE["catboost"]["test_by_seed"]
_pairs = [np.corrcoef(_cb[i], _cb[j])[0, 1] for i in range(5) for j in range(i + 1, 5)]
print(f"  {np.mean(_pairs):.5f}（5シード間の平均）")
print()
for b in names[1:]:
    c = corr.loc["catboost", b]
    if c > 0.99:
        print(f"  ⚠️ catboost vs {b}: 相関 {c:.5f} > 0.99 → 実質同じモデル。平均しても効果は望み薄")
    else:
        print(f"  ✅ catboost vs {b}: 相関 {c:.5f} → 多様性あり")


[2026-08-12 11:39:00] [INFO] ============================================================


INFO:43_model_diversity_equal_weight:============================================================


[2026-08-12 11:39:00] [INFO] [catboost] 単体学習（検証8シード + 全件5シード）


INFO:43_model_diversity_equal_weight:[catboost] 単体学習（検証8シード + 全件5シード）


[2026-08-12 11:39:31] [INFO]   val シード平均 0.514642 / 単一 0.517727 ± 0.005220


...

[2026-08-12 11:39:43] [INFO] [xgb] 単体学習（検証8シード + 全件5シード）


INFO:43_model_diversity_equal_weight:[xgb] 単体学習（検証8シード + 全件5シード）


[2026-08-12 11:39:51] [INFO]   val シード平均 0.541109 / 単一 0.544724 ± 0.004277


INFO:43_model_diversity_equal_weight:  val シード平均 0.541109 / 単一 0.544724 ± 0.004277


model          val(8シード)       単一平均      単一sd   CatBoost差     ゲート
------------------------------------------------------------------
catboost        0.514642   0.517727  0.005220   +0.000000       ✅
lgbm            0.546655   0.547901  0.000583   +0.032012       ❌
xgb             0.541109   0.544724  0.004277   +0.026466       ❌

ゲート通過: ['catboost']

多様性チェック: ファミリ間のTest予測相関
          catboost     lgbm      xgb
catboost   1.00000  0.92431  0.93809
lgbm       0.92431  1.00000  0.98157
xgb        0.93809  0.98157  1.00000

参考: CatBoost内でシードを引き直したときの相関（多様性ゼロの基準）
  0.98575（5シード間の平均）

  ✅ catboost vs lgbm: 相関 0.92431 → 多様性あり
  ✅ catboost vs xgb: 相関 0.93809 → 多様性あり


## B. 等重み平均

In [26]:
# ============================================================
# 第B節: 等重み平均（重みは学習しない）
# ============================================================

RESULT_SCHEMA = ["config", "members", "n_models", "val_seedavg", "val_single_mean",
                 "val_single_sd", "pred_mean", "mad_vs_E0", "corr_vs_E0", "submission_path"]


def make_ensemble(label, members):
    """メンバーのTest予測・検証予測を等重み平均する（確率の単純平均）"""
    v = np.mean([SINGLE[m]["val_preds"] for m in members], axis=0)
    t = np.mean([SINGLE[m]["test_preds"] for m in members], axis=0)
    base = SINGLE["catboost"]["test_preds"]
    path = save_submission(test_features_full.index, t, label)
    return make_row(config=label, members=",".join(members), n_models=len(members),
                    val_seedavg=float(log_loss(yva, v)),
                    val_single_mean=np.nan, val_single_sd=np.nan,
                    pred_mean=float(t.mean()),
                    mad_vs_E0=float(np.abs(t - base).mean()),
                    corr_vs_E0=float(np.corrcoef(t, base)[0, 1]),
                    submission_path=path)


CONFIGS = {"E0_catboost": ["catboost"]}
if "lgbm" in GATE_PASS:
    CONFIGS["E1_lgbm_only"] = ["lgbm"]
    CONFIGS["E3_cat_lgbm"] = ["catboost", "lgbm"]
if "xgb" in GATE_PASS:
    CONFIGS["E2_xgb_only"] = ["xgb"]
if len(GATE_PASS) >= 3:
    CONFIGS["E4_all3"] = GATE_PASS
elif len(GATE_PASS) == 2 and "xgb" in GATE_PASS and "lgbm" not in GATE_PASS:
    CONFIGS["E3b_cat_xgb"] = GATE_PASS

results = {n: make_ensemble(n, m) for n, m in CONFIGS.items()}

rows = pd.DataFrame(list(results.values()))
print(rows[["config", "members", "val_seedavg", "pred_mean", "mad_vs_E0", "corr_vs_E0"]]
      .round(6).to_string(index=False))
rows.to_csv(CHECKPOINT_PATH, index=False)
logger.info(f"結果を保存: {CHECKPOINT_PATH.name}")


[2026-08-12 11:39:51] [INFO]   提出ファイル: 20260812_43_model_diversity_equal_weight_E0_catboost.csv（予測平均=0.5874）


INFO:43_model_diversity_equal_weight:  提出ファイル: 20260812_43_model_diversity_equal_weight_E0_catboost.csv（予測平均=0.5874）


     config  members  val_seedavg  pred_mean  mad_vs_E0  corr_vs_E0
E0_catboost catboost     0.514642   0.587364        0.0         1.0
[2026-08-12 11:39:51] [INFO] 結果を保存: 43_model_diversity_equal_weight_checkpoint.csv


INFO:43_model_diversity_equal_weight:結果を保存: 43_model_diversity_equal_weight_checkpoint.csv


In [27]:
# ============================================================
# E0 の再現性チェック
#   E0 は 40_ R6_lean（Public 0.521729）と同じ特徴量・パラメータ・反復数・シード。
# ============================================================

_e0 = pd.read_csv(results["E0_catboost"]["submission_path"], header=None, names=[ID_COL, "pred"])
_r6 = sorted((PROJECT_ROOT / "data" / "output").glob("*/*_40_feature_reduction_R6_lean.csv"))
if _r6:
    _r = pd.read_csv(_r6[-1], header=None, names=[ID_COL, "pred"])
    _m = _e0.merge(_r, on=ID_COL, suffixes=("_e0", "_r6"))
    assert len(_m) == len(_e0), "社員IDが一致しない"
    print(f"R6_leanファイル: {_r6[-1].name}")
    print(f"  相関       : {_m['pred_e0'].corr(_m['pred_r6']):.6f}")
    print(f"  平均絶対差 : {(_m['pred_e0'] - _m['pred_r6']).abs().mean():.6f}")
    if _m["pred_e0"].corr(_m["pred_r6"]) > 0.9999 and (_m["pred_e0"] - _m["pred_r6"]).abs().mean() < 0.001:
        print("✅ R6_lean(Public 0.521729)を再現できている")
    else:
        print("⚠️ 再現できていない。前処理（-999埋め等）の差分を確認すること")
else:
    print("⚠️ R6_leanの提出ファイルが見つからなかった")


R6_leanファイル: 20260811_40_feature_reduction_R6_lean.csv
  相関       : 1.000000
  平均絶対差 : 0.000000
✅ R6_lean(Public 0.521729)を再現できている


## C. 提出判定

In [28]:
# ============================================================
# 第C節: 提出判定
# ============================================================

summary = pd.DataFrame(list(results.values()))
summary["提出"] = "見送り"
summary.loc[summary["config"] == "E0_catboost", "提出"] = "不要（提出済み・参照用）"
summary.loc[summary["config"].isin(["E1_lgbm_only", "E2_xgb_only"]), "提出"] = \
    "見送り（単体は事前登録どおり提出しない）"

_ens = summary["config"].str.startswith(("E3", "E4"))
summary.loc[_ens & (summary["mad_vs_E0"] > NOISE_FLOOR_P95), "提出"] = "提出する"
summary.loc[_ens & (summary["mad_vs_E0"] <= NOISE_FLOOR_P95), "提出"] = \
    "見送り（E0との差がノイズ床以下）"

pd.set_option("display.width", 220)
print(summary[["config", "members", "val_seedavg", "pred_mean", "mad_vs_E0", "corr_vs_E0", "提出"]]
      .round(6).to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)

print()
print("=" * 72)
print("提出候補")
print("=" * 72)
_sub = summary[summary["提出"] == "提出する"].sort_values("n_models")
if len(_sub) == 0:
    print("  なし。")
    print("  → アンサンブルの予測が CatBoost単体からノイズ床以上に離れなかった。")
    print("     モデルファミリを変えても同じ関数に収束しており、この手も尽きたと結論できる。")
else:
    for i, r in enumerate(_sub.itertuples(), 1):
        print(f"{i}. {r.config:<16s} {Path(r.submission_path).name}")
        print(f"     メンバー {r.members} / E0との平均絶対差 {r.mad_vs_E0:.5f} "
              f"/ 相関 {r.corr_vs_E0:.5f} / 予測平均 {r.pred_mean:.4f}")
print()
print(f"※ ノイズ床（41_実測）: 平均 {NOISE_FLOOR_MEAN} / 95%上限 {NOISE_FLOOR_P95}")


     config  members  val_seedavg  pred_mean  mad_vs_E0  corr_vs_E0           提出
E0_catboost catboost     0.514642   0.587364        0.0         1.0 不要（提出済み・参照用）

提出候補
  なし。
  → アンサンブルの予測が CatBoost単体からノイズ床以上に離れなかった。
     モデルファミリを変えても同じ関数に収束しており、この手も尽きたと結論できる。

※ ノイズ床（41_実測）: 平均 0.01246 / 95%上限 0.02122


## D. 提出方針と結果の解釈

### 提出するファイル

第C節で「提出する」となったもの（`E3_cat_lgbm` → `E4_all3` の順）。

- `E0_catboost` は `40_` R6_lean と同一内容なので**提出しない**
- `E1_lgbm_only`・`E2_xgb_only` は**事前登録どおり提出しない**。
  CatBoost単体に勝つことは期待しておらず、アンサンブルの材料としての質を測るために学習しただけ

### 結果の解釈ルール（事前登録）

- **Public < 0.521729** → モデル多様性は有効。次はメンバーや重みの粒度を検討する
  （ただし**重みの学習はしない**。[ensemble-oof-overfitting] の教訓は生きている）
- **Public ≒ 0.5217 ± 0.001** → 差なし。**CatBoost単体を基準構成のまま維持する**（軽い方が良い）
- **Public > 0.5217** → 多様性は害。この線を打ち切る

### 提出候補が0件だった場合

それ自体が結論である。**ファミリを変えても予測が同じ場所に収束するなら、
モデル側の自由度は使い切っている**ことになる。
`42_` でハイパーパラメータ再探索が「別の関数ではなく同じ関数への別の経路」だったのと同じ構図であり、
2つ合わせて「このデータでこの特徴量なら、到達できる関数は一つ」という結論になる。

### やらないこと

- **重みを学習しない。** `12_` の Stacking / Optimized Weighted は OOF を上回って Public で負け、
  `13_` の AutoGluon Weighted Ensemble（貪欲法）は同じ実行の CatBoost単体に 0.0012 負けている
- **検証スコアで提出構成を選び直さない。** 分解能±0.011に対し見込む効果は0.000〜0.004
- **ロジット平均は試さない。** 確率の単純平均はシード平均と同じ形式で Public 検証済み。
  ここで平均の取り方まで変えると、効いた要因が分離できなくなる

### 関連

- 現最良: `data/output/20260811/20260811_40_feature_reduction_R6_lean.csv`（Public 0.521729）
- 打った手の一覧: `submit_result_report.md` 第65節
- アンサンブルの過去の失敗: 同 第8〜10節（`12_`・`13_`）
